# Showing a result

Sixty-one public types in axiom report an outcome. Until this subpackage they all
printed the same way — a pydantic repr, three hundred characters on one line, with
the number you wanted somewhere in the middle.

A result that is hard to read gets skimmed, and a skimmed interval is a point
estimate. This is the layer that fixes that.

In [ ]:
from axiom.core import Assumption, Interval, LedgerLine, Summary, Unsupported, Unverified
from axiom.display import (
    REGISTRY,
    STATUS_MARK,
    STATUS_STYLE,
    Card,
    Row,
    Status,
    available,
    card_for,
    enable,
    generic_card,
    render,
    renders,
    show,
    status_from,
)
from axiom.identify import CausalGraph, identify, ols

print("rich installed:", available())

## 1. `show` — the one call

`show` prints a card. With `rich` installed it is a panel with a colour for its
status; without it, the same content as aligned plain text. `render` gives you
that text directly, which is what the tests assert on and what a log file wants.

In [ ]:
graph = CausalGraph.from_edges(
    "age -> dose, age -> pressure, dose -> adherence, adherence -> pressure",
    name="HYPER-3",
)
verdict = identify(graph, "dose", "pressure")
show(verdict, plain=True)

The status is the one piece of semantics a card carries, because it is the one
thing a reader takes at a glance: did this hold, is it assumed, did it fail.

In [ ]:
for state, mark in STATUS_MARK.items():
    print(f"{state:9s} {mark or '(nothing)':12s} -> {STATUS_STYLE[state]}")

## 2. An interval keeps its definition and its mass

The reporting half of rule 4: a point estimate that lost its interval on the way
to a terminal is the failure this package exists to prevent.

In [ ]:
band = Interval(lower=-16.7, upper=-8.1, definition="eti", mass=0.9)
show(band, plain=True)
print()
print(render(Summary(mean=-12.4, median=-12.3, sd=2.6, interval=band, n=2000)))

## 3. Failures render too

An `Unsupported` says what is missing and an `Unverified` says what was assumed.
Both are results, and both are worth reading rather than stringifying.

In [ ]:
show(Unsupported(reason="plotly is not installed", missing=("plotly",)), plain=True)
print()
show(Unverified(reason="the sampler did not converge"), plain=True)

## 4. Assumptions and the ledger

`satisfied` is green, `violated` is red, and everything else is *assumed* —
which is the honest colour for a condition nobody checked.

In [ ]:
assumption = Assumption(
    name="no_unmeasured_confounding",
    facet="population",
    statement="age is the only common cause of dose and pressure",
    challenged_by="a sensitivity analysis at plausible confounder strength",
    state="unverified",
)
show(assumption, plain=True)
print()
show(
    LedgerLine(
        kind="assumption",
        statement="Adherence was not adjusted for: it is a mediator.",
        assumption=assumption,
    ),
    plain=True,
)

## 5. An estimate leads with the number you came for

`emphasis` on a `Row` is what makes the estimate bold and the standard error not.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 400
age = rng.normal(size=n)
dose = 0.6 * age + rng.normal(size=n)
pressure = -1.2 * dose + 0.8 * age + rng.normal(size=n)
frame = pd.DataFrame({"age": age, "dose": dose, "pressure": pressure})

estimate = ols(frame, "pressure", "dose", ["age"])
show(estimate, plain=True)

## 6. Everything else still renders

There are far more result types than there are hand-written renderers, so every
`Spec` falls back to a card built from its own fields. `card_for` is the dispatch
and `generic_card` is that fallback — which is why adding this layer improved
sixty-one types rather than the dozen with a renderer of their own.

In [ ]:
from axiom.identify import TransportVerdict

print("types with a renderer of their own:", len(REGISTRY))
print()
print(render(generic_card(graph))[:400])

## 7. Adding one for your own type

`renders` registers a renderer. A `Card` is a title, a status, some `Row`s and a
note — the two back-ends only choose a typeface for it, so a renderer never has
to know whether colour is available.

In [ ]:
class Readout:
    def __init__(self, name: str, value: float) -> None:
        self.name, self.value = name, value


@renders(Readout)
def _readout(obj: Readout) -> Card:
    card = Card(title=f"Readout — {obj.name}", status="good")
    card.rows.append(Row("value", f"{obj.value:.2f}", emphasis=True))
    return card


show(Readout("retention", 0.797), plain=True)
print()
print("dispatch found:", card_for(Readout("x", 1.0)).title)

### Mapping your own vocabulary onto a status

A renderer usually has a word of its own — `satisfied`, `converged`, `stopped` —
and has to say which of the four statuses it means. `status_from` is that lookup.
It exists because `mapping.get(word, default)` widens the literal type away to
`str`, which is correct for a type checker and useless to every caller.

In [ ]:
outcomes: dict[str, Status] = {"converged": "good", "diverged": "bad"}
for word in ("converged", "diverged", "still running"):
    print(f"{word:15s} -> {status_from(word, outcomes, 'assumed')}")

## 8. In a notebook, `enable()` once

It registers a formatter with IPython, so every axiom result renders as a card
from then on with no `show` in front of it. A formatter rather than a
`_repr_html_` on the types themselves: those live in layers below this one and
must not learn that a display layer exists.

In [ ]:
status: Status = "good"
print("status vocabulary:", status)
print("formatter registered:", enable())
verdict